# Michigan statewide assessment data: requested extracts

This notebook reads the Michigan grades 3-8/PSAT and high-school assessment data files and creates analysis-ready CSV extracts for:

1. Statewide grades 3-8 proficiency in math, reading/ELA, science and social studies
2. Statewide eighth-grade math and third-grade reading/ELA proficiency by `ReportCategory`
3. Statewide eighth-grade PSAT scores and benchmark proficiency for all subjects
4. Statewide SAT scores overall and by subject
5. The same requested statewide extracts for Detroit Public Schools Community District (DPSCD)
6. District-level grades 3-8 proficiency, eighth-grade PSAT results and SAT results for every Michigan district

The code uses the field names in the two supplied layout workbooks. It is deliberately defensive about common differences in source files, including CSV vs. Excel, label capitalization, suppressed values, grade labels, ELA vs. reading terminology, and statewide/district row identification.

## Important interpretation

For M-STEP-style performance levels, `proficient_or_above_pct` is calculated as `PercentAdvanced + PercentProficient`. The original component fields are retained. For PSAT/SAT benchmark results, `PercentMet` and `PercentDidNotMeet` are retained. No school-level rows are aggregated, which avoids double-counting and preserves the state's published statewide and district records.

In [34]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# ---------------------------
# USER SETTINGS
# ---------------------------
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Change these to the actual assessment data filenames.
FILE_3_8 = DATA_DIR / "assessments_3_8.csv"
FILE_HS = DATA_DIR / "assessments_hs.csv"
FILE_COL_READY = DATA_DIR / "assessments_college_ready.csv"

# Use a specific year such as 2025, or None to keep every year in the data.
SCHOOL_YEAR = None

# DPSCD can be selected by district code or name. Keeping both makes the filter robust.
DPSCD_DISTRICT_CODE = "82015"
DPSCD_NAME_PATTERN = r"Detroit Public Schools Community District"

# If auto-detection does not identify published statewide rows correctly, set a known
# statewide label or code after reviewing the diagnostics cells below.
STATEWIDE_NAME_PATTERN = r"^(Michigan|Statewide|State Total)$"

## 1. Load and standardize the source files

The loader supports `.csv`, `.txt`, `.xlsx`, `.xls` and `.parquet`. Identifier fields are read as strings so leading zeroes are preserved. Suppression markers such as `*`, `--`, `<5` and blank strings become missing numeric values while the raw source columns remain available.

In [35]:
ID_COLS = ["ISDCode", "DistrictCode", "BuildingCode", "CountyCode"]
NUMERIC_COLS = [
    "TotalAdvanced", "TotalProficient", "TotalPartiallyProficient", "TotalNotProficient",
    "TotalSurpassed", "TotalAttained", "TotalEmergingTowards", "TotalMet", "TotalDidNotMeet",
    "PercentAdvanced", "PercentProficient", "PercentPartiallyProficient", "PercentNotProficient",
    "PercentSurpassed", "PercentAttained", "PercentEmergingTowards", "PercentMet",
    "PercentDidNotMeet", "AvgSS", "StdDevSS", "NumberAssessed", "MeanPtsEarned",
    "MinScaleScore", "MaxScaleScore", "ScaleScore25", "ScaleScore50", "ScaleScore75"
]

EXPECTED_COMMON = {
    "SchoolYear", "TestType", "TestPopulation", "ISDCode", "ISDName", "DistrictCode",
    "DistrictName", "BuildingCode", "BuildingName", "EntityType", "GradeContentTested",
    "Subject", "ReportCategory", "NumberAssessed"
}


def read_assessment(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Update FILE_3_8/FILE_HS or put the data file in {DATA_DIR.resolve()}."
        )
    suffix = path.suffix.lower()
    if suffix in {".csv", ".txt"}:
        df = pd.read_csv(path, dtype=str, low_memory=False)
    elif suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path, dtype=str, engine="openpyxl" if suffix == ".xlsx" else None)
    elif suffix == ".parquet":
        df = pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")

    df.columns = df.columns.astype(str).str.strip()
    missing = sorted(EXPECTED_COMMON - set(df.columns))
    if missing:
        raise ValueError(f"{path.name} is missing expected columns: {missing}")

    # Preserve identifier zeroes and strip spreadsheet-style trailing .0 if present.
    for col in ID_COLS:
        if col in df:
            df[col] = (df[col].astype("string").str.strip()
                       .str.replace(r"\.0$", "", regex=True)
                       .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA}))

    # Parse numeric fields without altering the original source vocabulary elsewhere.
    for col in set(NUMERIC_COLS).intersection(df.columns):
        cleaned = (df[col].astype("string").str.strip()
                   .str.replace(",", "", regex=False)
                   .str.replace("%", "", regex=False)
                   .replace({"": pd.NA, "*": pd.NA, "--": pd.NA, "—": pd.NA,
                             "<5": pd.NA, "< 5": pd.NA, "N/A": pd.NA, "NA": pd.NA}))
        df[col] = pd.to_numeric(cleaned, errors="coerce")

    # Normalized helper fields are only used for matching.
    for col in ["DistrictName", "BuildingName", "EntityType", "GradeContentTested",
                "Subject", "ReportCategory", "TestType", "TestPopulation"]:
        if col in df:
            df[f"_{col}"] = (df[col].astype("string").fillna("").str.strip().str.casefold())

    return df


df_3_8 = read_assessment(FILE_3_8)
df_hs = read_assessment(FILE_HS)
print(f"Grades 3-8/PSAT rows: {len(df_3_8):,}")
print(f"High-school rows:     {len(df_hs):,}")

ValueError: assessments_college_ready.csv is missing expected columns: ['GradeContentTested', 'NumberAssessed', 'ReportCategory', 'Subject', 'TestPopulation', 'TestType']

## 2. Inspect source vocabulary before filtering

Run these diagnostics first. They expose the exact year, assessment, grade, subject, report-category and entity labels in the delivered data. This is especially useful if the state changes labels between releases.

In [3]:
def values_report(df, columns, n=50):
    for col in columns:
        if col in df:
            values = df[col].dropna().astype(str).str.strip().value_counts().head(n)
            print(f"\n--- {col} ({df[col].nunique(dropna=True):,} unique) ---")
            display(values.rename("rows").to_frame())

values_report(df_3_8, ["SchoolYear", "TestType", "TestPopulation", "GradeContentTested",
                              "Subject", "ReportCategory", "EntityType"])
values_report(df_hs, ["SchoolYear", "TestType", "TestPopulation", "GradeContentTested",
                            "Subject", "ReportCategory", "EntityType"])


--- SchoolYear (1 unique) ---


,rows
SchoolYear,
24 - 25 School Year,551905



--- TestType (3 unique) ---


,rows
TestType,
M-STEP,377993
MI-Access,127549
PSAT,46363



--- TestPopulation (3 unique) ---


,rows
TestPopulation,
FI,61252
SI,37082
P,29215


... (18 outputs left out)


--- ReportCategory (16 unique) ---


,rows
ReportCategory,
All Students,14512
Not English Learners,14419
Male,13807
Economically Disadvantaged,13654
White,13305
Female,12375
Not Economically Disadvantaged,12122
Students Without Disabilities,10016
Students With Disabilities,9593



--- EntityType (11 unique) ---


,rows
EntityType,
LEA School,68707
LEA District,47897
PSA School,10033
PSA District,9379
ISD,8922
ISD School,4422
ISD District,3530
LEA Unique Education Provider,305
State District,88


## 3. Selection helpers and quality checks

The geography logic selects records already published at the relevant level:

- **Statewide:** rows explicitly labeled Michigan, Statewide or State Total, plus rows with no ISD, district or building identifiers
- **District:** rows with a district identifier and no building identifier
- **DPSCD:** district-level rows matching the DPSCD code or name

The notebook prints warnings if a requested extract is empty or appears to contain duplicate logical records.

In [24]:
def norm(series):
    return series.astype("string").fillna("").str.strip().str.casefold()


def apply_year(df):
    if SCHOOL_YEAR is None:
        return df.copy()
    return df.loc[norm(df["SchoolYear"]) == str(SCHOOL_YEAR).casefold()].copy()


def is_statewide(df):
    blank_codes = pd.Series(True, index=df.index)
    for col in ["ISDCode", "DistrictCode", "BuildingCode"]:
        if col in df:
            blank_codes &= df[col].isna() | norm(df[col]).eq("")

    label_match = pd.Series(False, index=df.index)
    for col in ["DistrictName", "ISDName", "BuildingName", "EntityType"]:
        if col in df:
            label_match |= df[col].astype("string").fillna("").str.strip().str.match(
                STATEWIDE_NAME_PATTERN, case=False, na=False
            )
    return blank_codes | label_match


def is_district_level(df):
    district_present = df["DistrictCode"].notna() & norm(df["DistrictCode"]).ne("")
    building_blank = df["BuildingCode"].isna() | norm(df["BuildingCode"]).eq("00000")
    return district_present & building_blank


def is_dpscd(df):
    code_match = norm(df["DistrictCode"]).eq(str(DPSCD_DISTRICT_CODE).casefold())
    name_match = df["DistrictName"].astype("string").fillna("").str.contains(
        DPSCD_NAME_PATTERN, case=False, regex=True, na=False
    )
    return is_district_level(df) & (code_match | name_match)


def grade_matches(series, grade):
    # Matches forms such as "3", "03", "Grade 3", "3rd Grade", and "8th".
    s = norm(series)
    number = str(int(grade))
    return s.str.contains(rf"(?<!\d)0?{number}(?!\d)", regex=True, na=False)

def subject_matches(series, subject):
    s = norm(series)
    patterns = {
        "math": r"math|mathematics",
        "reading": r"reading|ela|english language arts|\bela\b",
        "science": r"science",
        "social studies": r"social studies|social science",
    }
    return s.str.contains(patterns[subject], regex=True, na=False)


def test_matches(series, exam):
    s = norm(series)
    if exam == "psat8":
        return s.str.contains(r"psat", regex=True, na=False)
    if exam == "sat":
        return s.str.contains(r"(?<!p)\bsat\b|college board sat", regex=True, na=False)
    return pd.Series(True, index=series.index)


def overall_report_category(df):
    s = norm(df["ReportCategory"])
    known = s.str.contains(r"all students|overall|total", regex=True, na=False)
    # Some releases leave the all-students category blank.
    return known | s.eq("")


def add_proficiency(df):
    out = df.copy()
    advanced = out.get("PercentAdvanced", pd.Series(np.nan, index=out.index))
    proficient = out.get("PercentProficient", pd.Series(np.nan, index=out.index))
    out["proficient_or_above_pct"] = advanced.fillna(0) + proficient.fillna(0)
    out.loc[advanced.isna() & proficient.isna(), "proficient_or_above_pct"] = np.nan
    return out


OUTPUT_COLS = [
    "SchoolYear", "TestType", "TestPopulation", "ISDCode", "ISDName", "DistrictCode",
    "DistrictName", "EntityType", "GradeContentTested", "Subject", "ReportCategory",
    "NumberAssessed", "TotalAdvanced", "TotalProficient", "PercentAdvanced",
    "PercentProficient", "proficient_or_above_pct", "TotalMet", "TotalDidNotMeet",
    "PercentMet", "PercentDidNotMeet", "AvgSS", "StdDevSS", "MeanPtsEarned",
    "MinScaleScore", "MaxScaleScore", "ScaleScore25", "ScaleScore50", "ScaleScore75"
]


def finalize(df, name, keys):
    out = add_proficiency(df)
    cols = [c for c in OUTPUT_COLS if c in out.columns]
    out = out[cols].copy()
    sort_cols = [c for c in ["SchoolYear", "DistrictName", "GradeContentTested",
                              "Subject", "ReportCategory", "TestType"] if c in out]
    out = out.sort_values(sort_cols, kind="stable").reset_index(drop=True)
    if out.empty:
        warnings.warn(f"{name} is empty. Review the source vocabulary and filters above.")
    present_keys = [c for c in keys if c in out]
    if present_keys and out.duplicated(present_keys, keep=False).any():
        n = out.duplicated(present_keys, keep=False).sum()
        warnings.warn(f"{name}: {n:,} rows share the requested logical keys. Check TestPopulation, TestType and ReportCategory.")
    print(f"{name}: {len(out):,} rows")
    return out

## 4. Grades 3-8 proficiency extracts

For the broad grades 3-8 extracts, the notebook keeps the overall/all-students report category. The separate demographic extracts retain every `ReportCategory` for eighth-grade math and third-grade reading/ELA.

In [25]:
g38 = apply_year(df_3_8)
core_subject = (
    subject_matches(g38["Subject"], "math") |
    subject_matches(g38["Subject"], "reading") |
    subject_matches(g38["Subject"], "science") |
    subject_matches(g38["Subject"], "social studies")
)
grades_3_to_8 = pd.Series(False, index=g38.index)
for grade in range(3, 9):
    grades_3_to_8 |= grade_matches(g38["GradeContentTested"], grade)

base_g38 = grades_3_to_8 & core_subject & overall_report_category(g38)

g38_statewide = finalize(
    g38.loc[base_g38 & is_statewide(g38)],
    "grades_3_8_proficiency_statewide",
    ["SchoolYear", "GradeContentTested", "Subject", "ReportCategory", "TestType"]
)

g38_dpscd = finalize(
    g38.loc[base_g38 & is_dpscd(g38)],
    "grades_3_8_proficiency_dpscd",
    ["SchoolYear", "DistrictCode", "GradeContentTested", "Subject", "ReportCategory", "TestType"]
)

g38_all_districts = finalize(
    g38.loc[base_g38 & is_district_level(g38)],
    "grades_3_8_proficiency_all_districts",
    ["SchoolYear", "DistrictCode", "GradeContentTested", "Subject", "ReportCategory", "TestType"]
)

/var/folders/yn/1kz4ntb144195kbxp2lt33tc0000gn/T/ipykernel_89772/1300812868.py:104: UserWarning: grades_3_8_proficiency_statewide: 42 rows share the requested logical keys. Check TestPopulation, TestType and ReportCategory.
  warnings.warn(f"{name}: {n:,} rows share the requested logical keys. Check TestPopulation, TestType and ReportCategory.")


grades_3_8_proficiency_statewide: 60 rows


/var/folders/yn/1kz4ntb144195kbxp2lt33tc0000gn/T/ipykernel_89772/1300812868.py:104: UserWarning: grades_3_8_proficiency_dpscd: 42 rows share the requested logical keys. Check TestPopulation, TestType and ReportCategory.
  warnings.warn(f"{name}: {n:,} rows share the requested logical keys. Check TestPopulation, TestType and ReportCategory.")


grades_3_8_proficiency_dpscd: 60 rows
grades_3_8_proficiency_all_districts: 23,339 rows


/var/folders/yn/1kz4ntb144195kbxp2lt33tc0000gn/T/ipykernel_89772/1300812868.py:104: UserWarning: grades_3_8_proficiency_all_districts: 7,761 rows share the requested logical keys. Check TestPopulation, TestType and ReportCategory.
  warnings.warn(f"{name}: {n:,} rows share the requested logical keys. Check TestPopulation, TestType and ReportCategory.")


In [28]:
requested_demo_tests = (
    (grade_matches(g38["GradeContentTested"], 8) & subject_matches(g38["Subject"], "math")) |
    (grade_matches(g38["GradeContentTested"], 3) & subject_matches(g38["Subject"], "reading"))
)

g38_report_category_statewide = finalize(
    g38.loc[requested_demo_tests & is_statewide(g38)],
    "grade_subject_report_categories_statewide",
    ["SchoolYear", "GradeContentTested", "Subject", "ReportCategory", "TestType"]
)

g38_report_category_dpscd = finalize(
    g38.loc[requested_demo_tests & is_dpscd(g38)],
    "grade_subject_report_categories_dpscd",
    ["SchoolYear", "DistrictCode", "GradeContentTested", "Subject", "ReportCategory", "TestType"]
)

/var/folders/yn/1kz4ntb144195kbxp2lt33tc0000gn/T/ipykernel_89772/1300812868.py:104: UserWarning: grade_subject_report_categories_statewide: 77 rows share the requested logical keys. Check TestPopulation, TestType and ReportCategory.
  warnings.warn(f"{name}: {n:,} rows share the requested logical keys. Check TestPopulation, TestType and ReportCategory.")


grade_subject_report_categories_statewide: 109 rows
grade_subject_report_categories_dpscd: 93 rows


/var/folders/yn/1kz4ntb144195kbxp2lt33tc0000gn/T/ipykernel_89772/1300812868.py:104: UserWarning: grade_subject_report_categories_dpscd: 62 rows share the requested logical keys. Check TestPopulation, TestType and ReportCategory.
  warnings.warn(f"{name}: {n:,} rows share the requested logical keys. Check TestPopulation, TestType and ReportCategory.")


## 5. Eighth-grade PSAT extracts

These extracts retain all subjects and the available score-distribution fields, along with `PercentMet` and `PercentDidNotMeet`. The overall/all-students report category is used to prevent demographic duplication.

In [29]:
psat8_mask = (
    grade_matches(g38["GradeContentTested"], 8) &
    test_matches(g38["TestType"], "psat8") &
    overall_report_category(g38)
)

psat8_statewide = finalize(
    g38.loc[psat8_mask & is_statewide(g38)],
    "psat8_statewide",
    ["SchoolYear", "GradeContentTested", "Subject", "ReportCategory", "TestType"]
)

psat8_dpscd = finalize(
    g38.loc[psat8_mask & is_dpscd(g38)],
    "psat8_dpscd",
    ["SchoolYear", "DistrictCode", "GradeContentTested", "Subject", "ReportCategory", "TestType"]
)

psat8_all_districts = finalize(
    g38.loc[psat8_mask & is_district_level(g38)],
    "psat8_all_districts",
    ["SchoolYear", "DistrictCode", "GradeContentTested", "Subject", "ReportCategory", "TestType"]
)

psat8_statewide: 2 rows
psat8_dpscd: 2 rows
psat8_all_districts: 1,678 rows


/var/folders/yn/1kz4ntb144195kbxp2lt33tc0000gn/T/ipykernel_89772/1300812868.py:104: UserWarning: psat8_all_districts: 116 rows share the requested logical keys. Check TestPopulation, TestType and ReportCategory.
  warnings.warn(f"{name}: {n:,} rows share the requested logical keys. Check TestPopulation, TestType and ReportCategory.")


## 6. SAT score extracts

These are actually in a separate file with a slightly different  format. We extract them from the separate file.

In [36]:
df_sat = pd.read_csv(FILE_COL_READY)

In [50]:
#extract state-level SAT scores
sat_statewide = df_sat[df_sat['ISDName'] == 'Statewide']
sat_statewide.head()

,SchoolYear,ISDCode,ISDName,DistrictCode,DistrictName,BuildingCode,BuildingName,CountyCode,CountyName,EntityType,SchoolLevel,Locale,MISTEM_NAME,MISTEM_CODE,Subgroup,MathPercentReady,MathNumAssessed,MathScoreAverage,MathCountReady,ReadingPercentReady,ReadingNumAssessed,ReadingScoreAverage,SciencePercentReady,ScienceNumAssessed,ScienceScoreAverage,EnglishPercentReady,EnglishNumAssessed,EnglishScoreAverage,AllSubjectPercentReady,AllSubjectNumAssessed,AllSubjectScoreAverage,AllCountReady,EBRWPercentReady,EBRWNumAssessed,EBRWScoreAverage,EBRWCountReady
0,24 - 25 School Year,0,Statewide,0,All Districts,0,All Buildings,NaN,NaN,State,NaN,NaN,NaN,NaN,All Students,27.5000,97412,463.8000,26790,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.6000,97340,954.0000,25870,55.4000,97340,490.1000,53886
1,24 - 25 School Year,0,Statewide,0,All Districts,0,All Buildings,NaN,NaN,State,NaN,NaN,NaN,NaN,American Indian or Alaska Native,15.7000,578,433.8000,91,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.6000,578,898.9000,90,47.4000,578,465.1000,274
2,24 - 25 School Year,0,Statewide,0,All Districts,0,All Buildings,NaN,NaN,State,NaN,NaN,NaN,NaN,Asian,61.7000,3926,576.4000,2424,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.7000,3924,1142.2000,2341,76.2000,3924,565.7000,2992
3,24 - 25 School Year,0,Statewide,0,All Districts,0,All Buildings,NaN,NaN,State,NaN,NaN,NaN,NaN,"Black, not of Hispanic origin",7.8000,15326,393.0000,1189,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.1000,15319,819.4000,1085,28.5000,15319,426.5000,4367
4,24 - 25 School Year,0,Statewide,0,All Districts,0,All Buildings,NaN,NaN,State,NaN,NaN,NaN,NaN,Economically Disadvantaged,12.9000,42522,417.5000,5495,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.1000,42484,865.7000,5132,38.5000,42484,448.1000,16367


In [51]:
#extract SAT scores for Detroit schools
sat_dpscd = df_sat[(df_sat['DistrictName'] == 'Detroit Public Schools Community District') & (df_sat["BuildingName"] == 'All Buildings')]

#filter to subgroups where >10 students were assessed for math and reading
sat_dpscd = sat_dpscd[(sat_dpscd["MathNumAssessed"] != "<10") & (sat_dpscd["ReadingNumAssessed"] != "<10")]
sat_dpscd.head()

,SchoolYear,ISDCode,ISDName,DistrictCode,DistrictName,BuildingCode,BuildingName,CountyCode,CountyName,EntityType,SchoolLevel,Locale,MISTEM_NAME,MISTEM_CODE,Subgroup,MathPercentReady,MathNumAssessed,MathScoreAverage,MathCountReady,ReadingPercentReady,ReadingNumAssessed,ReadingScoreAverage,SciencePercentReady,ScienceNumAssessed,ScienceScoreAverage,EnglishPercentReady,EnglishNumAssessed,EnglishScoreAverage,AllSubjectPercentReady,AllSubjectNumAssessed,AllSubjectScoreAverage,AllCountReady,EBRWPercentReady,EBRWNumAssessed,EBRWScoreAverage,EBRWCountReady
26586,24 - 25 School Year,82,Wayne RESA,82015,Detroit Public Schools Community District,0,All Buildings,82.0,Wayne,LEA District,Elem thru High School,City: Large,Wayne Region,3.0,All Students,14.2000,2874,411.9000,409,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.2000,2857,850.1000,378,36.1000,2857,437.8000,1030
26588,24 - 25 School Year,82,Wayne RESA,82015,Detroit Public Schools Community District,0,All Buildings,82.0,Wayne,LEA District,Elem thru High School,City: Large,Wayne Region,3.0,Asian,48.6000,74,514.9000,36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,48.6000,74,1045.5000,36,77.0000,74,530.7000,57
26589,24 - 25 School Year,82,Wayne RESA,82015,Detroit Public Schools Community District,0,All Buildings,82.0,Wayne,LEA District,Elem thru High School,City: Large,Wayne Region,3.0,"Black, not of Hispanic origin",13.8000,2255,411.8000,311,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.8000,2252,851.1000,288,36.4000,2252,439.2000,819
26590,24 - 25 School Year,82,Wayne RESA,82015,Detroit Public Schools Community District,0,All Buildings,82.0,Wayne,LEA District,Elem thru High School,City: Large,Wayne Region,3.0,Economically Disadvantaged,12.4000,2249,403.4000,279,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.4000,2244,831.8000,256,31.6000,2244,428.1000,709
26591,24 - 25 School Year,82,Wayne RESA,82015,Detroit Public Schools Community District,0,All Buildings,82.0,Wayne,LEA District,Elem thru High School,City: Large,Wayne Region,3.0,English Language Learners,5.9000,422,376.8000,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.2000,408,766.9000,17,16.4000,408,388.9000,67


In [52]:
#exctract SAT scores for all districts (just for all students, no subgroups)
sat_all_districts = df_sat[(df_sat['ISDName'] != 'Statewide') & (df_sat["BuildingName"] == 'All Buildings') & (df_sat["Subgroup"] == 'All Students')]
sat_all_districts.head()

,SchoolYear,ISDCode,ISDName,DistrictCode,DistrictName,BuildingCode,BuildingName,CountyCode,CountyName,EntityType,SchoolLevel,Locale,MISTEM_NAME,MISTEM_CODE,Subgroup,MathPercentReady,MathNumAssessed,MathScoreAverage,MathCountReady,ReadingPercentReady,ReadingNumAssessed,ReadingScoreAverage,SciencePercentReady,ScienceNumAssessed,ScienceScoreAverage,EnglishPercentReady,EnglishNumAssessed,EnglishScoreAverage,AllSubjectPercentReady,AllSubjectNumAssessed,AllSubjectScoreAverage,AllCountReady,EBRWPercentReady,EBRWNumAssessed,EBRWScoreAverage,EBRWCountReady
24,24 - 25 School Year,3,Allegan Area Educational Service Agency,0,All Districts,0,All Buildings,3.0,Allegan,ISD,Elem thru High School,Town: Distant,Greater West Michigan Region,8.0,All Students,28.0000,923,469.0000,258,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.9000,923,962.0000,248,58.0000,923,493.0000,535
48,24 - 25 School Year,3,Allegan Area Educational Service Agency,3000,Allegan Area Educational Service Agency - Dis...,0,All Buildings,3.0,Allegan,ISD District,Elem thru High School,Town: Distant,Greater West Michigan Region,8.0,All Students,*,<10,*,*,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,*,<10,*,*,*,<10,*,*
72,24 - 25 School Year,3,Allegan Area Educational Service Agency,3010,Plainwell Community Schools,0,All Buildings,3.0,Allegan,LEA District,Elem thru High School,Town: Fringe,Greater West Michigan Region,8.0,All Students,39.1000,197,488.5000,77,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.1000,197,988.8000,75,61.9000,197,500.4000,122
131,24 - 25 School Year,3,Allegan Area Educational Service Agency,3020,Otsego Public Schools,0,All Buildings,3.0,Allegan,LEA District,Elem thru High School,Town: Fringe,Greater West Michigan Region,8.0,All Students,37.4000,147,496.7000,55,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.7000,147,1025.9000,54,73.5000,147,529.1000,108
181,24 - 25 School Year,3,Allegan Area Educational Service Agency,3030,Allegan Public Schools,0,All Buildings,3.0,Allegan,LEA District,Elem thru High School,Town: Distant,Greater West Michigan Region,8.0,All Students,12.7000,157,440.1000,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.7000,157,916.8000,20,52.2000,157,476.6000,82


## 7. Review and export

The manifest provides a quick row-count check. Each requested subset is written to a separate CSV in `OUTPUT_DIR`. Empty extracts are still written, making missing selections visible and reproducible rather than silently omitted.

In [53]:
extracts = {
    "grades_3_8_proficiency_statewide": g38_statewide,
    "grade_subject_report_categories_statewide": g38_report_category_statewide,
    "psat8_statewide": psat8_statewide,
    "sat_statewide": sat_statewide,
    "grades_3_8_proficiency_dpscd": g38_dpscd,
    "grade_subject_report_categories_dpscd": g38_report_category_dpscd,
    "psat8_dpscd": psat8_dpscd,
    "sat_dpscd": sat_dpscd,
    "grades_3_8_proficiency_all_districts": g38_all_districts,
    "psat8_all_districts": psat8_all_districts,
    "sat_all_districts": sat_all_districts,
}

manifest = pd.DataFrame([
    {
        "extract": name,
        "rows": len(frame),
        "school_years": ", ".join(sorted(frame["SchoolYear"].dropna().astype(str).unique())) if "SchoolYear" in frame else "",
        "districts": frame["DistrictCode"].nunique(dropna=True) if "DistrictCode" in frame else 0,
        "subjects": ", ".join(sorted(frame["Subject"].dropna().astype(str).unique())) if "Subject" in frame else "",
    }
    for name, frame in extracts.items()
])
display(manifest)

for name, frame in extracts.items():
    frame.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
manifest.to_csv(OUTPUT_DIR / "extract_manifest.csv", index=False)

print(f"Wrote {len(extracts) + 1} CSV files to: {OUTPUT_DIR.resolve()}")

,extract,rows,school_years,districts,subjects
0,grades_3_8_proficiency_statewide,60,24 - 25 School Year,1,"ELA, Mathematics, Science, Social Studies"
1,grade_subject_report_categories_statewide,109,24 - 25 School Year,1,"ELA, Mathematics"
2,psat8_statewide,2,24 - 25 School Year,1,"ELA, Mathematics"
3,sat_statewide,24,24 - 25 School Year,1,
4,grades_3_8_proficiency_dpscd,60,24 - 25 School Year,1,"ELA, Mathematics, Science, Social Studies"
5,grade_subject_report_categories_dpscd,93,24 - 25 School Year,1,"ELA, Mathematics"
6,psat8_dpscd,2,24 - 25 School Year,1,"ELA, Mathematics"
7,sat_dpscd,20,24 - 25 School Year,1,
8,grades_3_8_proficiency_all_districts,23339,24 - 25 School Year,843,"ELA, Mathematics, Science, Social Studies"
9,psat8_all_districts,1678,24 - 25 School Year,782,"ELA, Mathematics"


Wrote 12 CSV files to: /Users/benwarren/Independent Projects/Detroit News Data/mstep_2026/outputs
